# F2LLM-v2-8B → Qdrant — Kaggle T4

Pipeline bám theo model card chính thức của F2LLM-v2:

- F2LLM-v2-8B dùng kiến trúc nền Qwen3; module `transformers.models.qwen3...` là đúng.
- Query có instruction, document không thêm instruction.
- Lấy hidden state tại token cuối/EOS.
- Chuyển sang float32 trước khi L2-normalize để tránh sai norm khi chạy 4-bit.
- Lưu vector 4096 chiều bằng cosine distance.

## 0. Cài thư viện

Chạy trong session Kaggle mới, sau đó **Restart Session**. Không cài `sentence-transformers`, tránh kéo theo SciPy/sklearn và tránh normalization ở precision thấp.

In [1]:
!pip -q install --no-deps --force-reinstall \
    "huggingface-hub==0.30.2" \
    "tokenizers==0.21.1" \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "bitsandbytes==0.49.2"
!pip -q install -U "qdrant-client>=1.10,<2"

print("Cài đặt xong. Hãy Restart Session rồi chạy từ cell tiếp theo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 12.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 58.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 91.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 7.6 MB/s eta 0:00:00a 0:00:01
Cài đặt xong. Hãy Restart Session rồi chạy từ cell tiếp theo.


## 1. Cấu hình, Qdrant và corpus

In [2]:
import gc
import glob
import json
import os

import accelerate
import bitsandbytes
import torch
import transformers
from kaggle_secrets import UserSecretsClient
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "codefuse-ai/F2LLM-v2-8B"
MODEL_REVISION = "e5725783762d69b4f8ba7e09a8872ce19a7a5ec3"
COLLECTION = "laws_f2llm"
EXPECTED_DIM = 4096
MAX_LENGTH = 8192
ENCODE_BATCH = 1
UPSERT_BATCH = 64
INSTRUCTION = "Given a legal question in Vietnamese, retrieve the most relevant law articles"

EXPECTED_VERSIONS = {
    "transformers": "4.51.3",
    "accelerate": "1.6.0",
    "bitsandbytes": "0.49.2",
}
actual_versions = {
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__,
}
print("Versions:", actual_versions)
assert actual_versions == EXPECTED_VERSIONS, (
    "Sai phiên bản. Hãy Restart Session sau cell cài đặt. "
    f"Expected={EXPECTED_VERSIONS}, actual={actual_versions}"
)
assert torch.cuda.is_available(), "Hãy bật GPU Kaggle."
print("CUDA:", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))

secrets = UserSecretsClient()
qdrant_url = secrets.get_secret("QDRANT_URL")
try:
    qdrant_key = secrets.get_secret("QDRANT_KEY")
except Exception:
    qdrant_key = secrets.get_secret("QDRANT_API_KEY")
try:
    hf_token = secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if not hf_token:
    print("HF_TOKEN chưa được đặt: vẫn tải được model public, chỉ có rate limit thấp hơn.")

qc = QdrantClient(url=qdrant_url, api_key=qdrant_key, timeout=300)
print("Qdrant collections:", [x.name for x in qc.get_collections().collections])

def find_corpus():
    candidates = ["/kaggle/working/corpus_law_pub.json", "corpus_law_pub.json"]
    candidates += glob.glob("/kaggle/input/**/corpus_law_pub.json", recursive=True)
    for path in candidates:
        if os.path.isfile(path):
            return path
    raise FileNotFoundError("Không tìm thấy corpus_law_pub.json")

corpus_path = find_corpus()
with open(corpus_path, encoding="utf-8") as f:
    corpus = json.load(f)

records = []
for law in corpus:
    for article_no, article in enumerate(law["content"], start=1):
        text = (article.get("content_Article") or "").strip()
        if not text:
            continue
        aid = int(article["aid"])
        records.append({
            "point_id": aid,
            "aid": aid,
            "law_id": law["law_id"],
            "article_no": article_no,
            "text": text,
        })

assert records
assert len({r['aid'] for r in records}) == len(records), "aid bị trùng"
print("Corpus:", corpus_path, "| laws:", len(corpus), "| articles:", len(records))

Versions: {'transformers': '4.51.3', 'accelerate': '1.6.0', 'bitsandbytes': '0.49.2'}
CUDA: 12.8 | GPUs: 2
  0 Tesla T4
  1 Tesla T4
HF_TOKEN chưa được đặt: vẫn tải được model public, chỉ có rate limit thấp hơn.
Qdrant collections: ['alqac_laws_raw', 'laws_gte_qwen2', 'laws_bge_m3_v2_correct_pooling', 'alqac_laws_test']
Corpus: /kaggle/input/datasets/ldhhieu18/corpus-law/corpus_law_pub.json | laws: 18 | articles: 3352


## 2. Load F2LLM 4-bit và hàm embedding

In [3]:
def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

# Phát hiện sớm lỗi bitsandbytes/CUDA.
_bnb_test = bitsandbytes.nn.Linear4bit(
    16, 16, bias=False, compute_dtype=torch.float16, quant_type="nf4"
).to("cuda:0")
with torch.inference_mode():
    _bnb_output = _bnb_test(torch.randn(2, 16, device="cuda:0", dtype=torch.float16))
assert bool(torch.isfinite(_bnb_output).all())
del _bnb_test, _bnb_output
clear_cuda()
print("bitsandbytes NF4 smoke test: OK")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=hf_token,
)
# Công thức eos_position = attention_mask.sum()-1 yêu cầu right padding.
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=hf_token,
    quantization_config=quantization,
    device_map="auto",
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.use_cache = False

model_module = type(model).__module__
hidden_size = int(model.config.hidden_size)
architecture_limit = int(model.config.max_position_embeddings)
print("Model module:", model_module)
print("Hidden size:", hidden_size, "| max positions:", architecture_limit)
print("Device map:", getattr(model, "hf_device_map", None))
assert "qwen3" in model_module.lower(), "F2LLM-v2-8B phải dùng kiến trúc Qwen3"
assert getattr(model, "is_loaded_in_4bit", False)
assert hidden_size == EXPECTED_DIM
assert architecture_limit >= MAX_LENGTH

def encode_texts(texts, batch_size=ENCODE_BATCH):
    if not texts:
        return torch.empty((0, EXPECTED_DIM), dtype=torch.float32)
    batch_size = max(1, int(batch_size))
    while True:
        try:
            all_vectors = []
            input_device = model.get_input_embeddings().weight.device
            for start in range(0, len(texts), batch_size):
                batch_texts = [(text or " ").strip() or " " for text in texts[start:start + batch_size]]
                inputs = tokenizer(
                    batch_texts,
                    max_length=MAX_LENGTH,
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                )
                inputs = {key: value.to(input_device) for key, value in inputs.items()}
                with torch.inference_mode():
                    outputs = model(**inputs, use_cache=False, return_dict=True)
                    eos_positions = inputs["attention_mask"].sum(dim=1) - 1
                    rows = torch.arange(len(batch_texts), device=outputs.last_hidden_state.device)
                    eos_positions = eos_positions.to(outputs.last_hidden_state.device)
                    vectors = outputs.last_hidden_state[rows, eos_positions]
                    # Quan trọng: normalize trong float32, không normalize trực tiếp fp16/4-bit output.
                    vectors = torch.nn.functional.normalize(vectors.float(), p=2, dim=1)
                all_vectors.append(vectors.cpu())
                done = min(start + batch_size, len(texts))
                if done % 100 == 0 or done == len(texts):
                    print("embedded", done, "/", len(texts))
                del inputs, outputs, vectors
            return torch.cat(all_vectors, dim=0)
        except torch.cuda.OutOfMemoryError:
            clear_cuda()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print("CUDA OOM, retry batch_size=", batch_size)

def make_query(text):
    return f"Instruct: {INSTRUCTION}\nQuery: {(text or '').strip()}"

probe = encode_texts([records[0]["text"], make_query("bồi thường do súc vật gây ra")], 1)
probe_norms = torch.linalg.vector_norm(probe, dim=1)
print("Probe:", tuple(probe.shape), "| norms:", probe_norms.tolist())
assert tuple(probe.shape) == (2, EXPECTED_DIM)
assert bool(torch.isfinite(probe).all())
assert torch.allclose(probe_norms, torch.ones_like(probe_norms), atol=1e-5)

bitsandbytes NF4 smoke test: OK


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/386M [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model module: transformers.models.qwen3.modeling_qwen3
Hidden size: 4096 | max positions: 40960
Device map: {'embed_tokens': 0, 'layers.0': 0, 'layers.1': 0, 'layers.2': 0, 'layers.3': 0, 'layers.4': 0, 'layers.5': 0, 'layers.6': 0, 'layers.7': 0, 'layers.8': 0, 'layers.9': 0, 'layers.10': 0, 'layers.11': 0, 'layers.12': 0, 'layers.13': 0, 'layers.14': 1, 'layers.15': 1, 'layers.16': 1, 'layers.17': 1, 'layers.18': 1, 'layers.19': 1, 'layers.20': 1, 'layers.21': 1, 'layers.22': 1, 'layers.23': 1, 'layers.24': 1, 'layers.25': 1, 'layers.26': 1, 'layers.27': 1, 'layers.28': 1, 'layers.29': 1, 'layers.30': 1, 'layers.31': 1, 'layers.32': 1, 'layers.33': 1, 'layers.34': 1, 'layers.35': 1, 'norm': 1, 'rotary_emb': 1}
embedded 2 / 2
Probe: (2, 4096) | norms: [1.0, 0.9999998807907104]


## 3. Embed document và lưu Qdrant

In [4]:
token_lengths = []
for i, record in enumerate(records, start=1):
    length = len(tokenizer(record["text"], add_special_tokens=True, truncation=False)["input_ids"])
    token_lengths.append(length)
    if i % 500 == 0 or i == len(records):
        print("token-count", i, "/", len(records))

long_indices = [i for i, length in enumerate(token_lengths) if length > MAX_LENGTH]
print("Token min/mean/max:", min(token_lengths), round(sum(token_lengths) / len(token_lengths), 2), max(token_lengths))
if long_indices:
    examples = [
        (records[i]["law_id"], records[i]["article_no"], records[i]["aid"], token_lengths[i])
        for i in long_indices[:10]
    ]
    raise ValueError(f"Có {len(long_indices)} article vượt MAX_LENGTH={MAX_LENGTH}: {examples}")

# Document không thêm instruction.
document_vectors = encode_texts([record["text"] for record in records], ENCODE_BATCH)
assert tuple(document_vectors.shape) == (len(records), EXPECTED_DIM)
assert bool(torch.isfinite(document_vectors).all())
norms = torch.linalg.vector_norm(document_vectors, dim=1)
print("Vectors:", tuple(document_vectors.shape), "| norm min/mean/max:", norms.min().item(), norms.mean().item(), norms.max().item())
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5)

if qc.collection_exists(COLLECTION):
    qc.delete_collection(COLLECTION)
qc.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EXPECTED_DIM, distance=Distance.COSINE),
)

for start in range(0, len(records), UPSERT_BATCH):
    end = min(start + UPSERT_BATCH, len(records))
    points = []
    for record, vector in zip(records[start:end], document_vectors[start:end]):
        points.append(PointStruct(
            id=record["point_id"],
            vector=vector.tolist(),
            payload={
                "aid": record["aid"],
                "law_id": record["law_id"],
                "article_no": record["article_no"],
                "content_Article": record["text"],
                "embedding_model": MODEL_ID,
                "embedding_revision": MODEL_REVISION,
                "embedding_pipeline": "f2llm_qwen3_last_eos_float32_l2_4bit",
                "transformers_version": transformers.__version__,
                "max_length": MAX_LENGTH,
            },
        ))
    qc.upsert(collection_name=COLLECTION, points=points, wait=True)
    print("upserted", end, "/", len(records))

info = qc.get_collection(COLLECTION)
stored_dim = int(info.config.params.vectors.size)
stored_count = int(qc.count(collection_name=COLLECTION, exact=True).count)
print("Collection:", COLLECTION, "| dim:", stored_dim, "| points:", stored_count)
assert stored_dim == EXPECTED_DIM
assert stored_count == len(records)

token-count 500 / 3352
token-count 1000 / 3352
token-count 1500 / 3352
token-count 2000 / 3352
token-count 2500 / 3352
token-count 3000 / 3352
token-count 3352 / 3352
Token min/mean/max: 19 248.3 2976
embedded 100 / 3352
embedded 200 / 3352
embedded 300 / 3352
embedded 400 / 3352
embedded 500 / 3352
embedded 600 / 3352
embedded 700 / 3352
embedded 800 / 3352
embedded 900 / 3352
embedded 1000 / 3352
embedded 1100 / 3352
embedded 1200 / 3352
embedded 1300 / 3352
embedded 1400 / 3352
embedded 1500 / 3352
embedded 1600 / 3352
embedded 1700 / 3352
embedded 1800 / 3352
embedded 1900 / 3352
embedded 2000 / 3352
embedded 2100 / 3352
embedded 2200 / 3352
embedded 2300 / 3352
embedded 2400 / 3352
embedded 2500 / 3352
embedded 2600 / 3352
embedded 2700 / 3352
embedded 2800 / 3352
embedded 2900 / 3352
embedded 3000 / 3352
embedded 3100 / 3352
embedded 3200 / 3352
embedded 3300 / 3352
embedded 3352 / 3352
Vectors: (3352, 4096) | norm min/mean/max: 0.9999997615814209 1.0 1.000000238418579
upserted 6

## 4. Kiểm tra alignment và retrieval

In [5]:
for index in sorted(set([0, len(records) // 2, len(records) - 1])):
    record = records[index]
    hits = qc.query_points(
        collection_name=COLLECTION,
        query=document_vectors[index].tolist(),
        limit=3,
        with_payload=True,
    ).points
    aids = [int((hit.payload or {})["aid"]) for hit in hits]
    assert record["aid"] in aids
    print("self-retrieval aid=", record["aid"], "rank=", aids.index(record["aid"]) + 1)

def search(query, top=5):
    query_vector = encode_texts([make_query(query)], 1)[0]
    hits = qc.query_points(
        collection_name=COLLECTION,
        query=query_vector.tolist(),
        limit=top,
        with_payload=True,
    ).points
    print("\nQuery:", query)
    for rank, hit in enumerate(hits, start=1):
        payload = hit.payload or {}
        snippet = (payload.get("content_Article") or "").replace("\n", " ")[:140]
        print(
            f"{rank:2d}. score={hit.score:.4f} | {payload.get('law_id')} "
            f"Điều {payload.get('article_no')} | aid={payload.get('aid')} | {snippet}..."
        )
    return hits

search("bồi thường thiệt hại do súc vật gây ra", top=5)
search("thời hiệu khởi kiện yêu cầu chia di sản thừa kế", top=5)
print("\n✓ F2LLM embedding và Qdrant pipeline đã hoàn tất.")

self-retrieval aid= 270 rank= 1
self-retrieval aid= 53219 rank= 1
self-retrieval aid= 57130 rank= 1
embedded 1 / 1

Query: bồi thường thiệt hại do súc vật gây ra
 1. score=0.8599 | 91/2015/QH13 Điều 603 | aid=53373 | 1. Chủ sở hữu súc vật phải bồi thường thiệt hại do súc vật gây ra cho người khác. Người chiếm hữu, sử dụng súc vật phải bồi thường thiệt hại...
 2. score=0.5449 | 91/2015/QH13 Điều 584 | aid=53354 | 1. Người nào có hành vi xâm phạm tính mạng, sức khỏe, danh dự, nhân phẩm, uy tín, tài sản, quyền, lợi ích hợp pháp khác của người khác mà gâ...
 3. score=0.4790 | 91/2015/QH13 Điều 604 | aid=53374 | Chủ sở hữu, người chiếm hữu, người được giao quản lý phải bồi thường thiệt hại do cây cối gây ra....
 4. score=0.4504 | 91/2015/QH13 Điều 586 | aid=53356 | 1. Người từ đủ mười tám tuổi trở lên gây thiệt hại thì phải tự bồi thường.  2. Người chưa đủ mười lăm tuổi gây thiệt hại mà còn cha, mẹ thì ...
 5. score=0.4386 | 91/2015/QH13 Điều 605 | aid=53375 | Chủ sở hữu, người chiếm hữu, n